In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

bronze_base = "abfss://bronze@travelappprojectstorage.dfs.core.windows.net"
silver_base = "abfss://silver@travelappprojectstorage.dfs.core.windows.net/trip_adherence"
quarantine_base = "abfss://quarantine@travelappprojectstorage.dfs.core.windows.net/trip_adherence"

In [0]:
trip_df = spark.read.format("parquet").load(
    f"{bronze_base}/trip"
)

trip_schedule_df = spark.read.format("parquet").load(
    f"{bronze_base}/trip_schedule"
)

tourist_df = spark.read.format("parquet").load(
    f"{bronze_base}/tourist"
)

agency_df = spark.read.format("parquet").load(
    f"{bronze_base}/agency"
)

trip_request_places_df = spark.read.format("parquet").load(
    f"{bronze_base}/trip_request_places"
)

tourist_place_df = spark.read.format("parquet").load(
    f"{bronze_base}/trip_places"
)

In [0]:
location_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option(
        "cloudFiles.schemaLocation",
        f"{bronze_base}/_schemas/trip_adherence_silver"
    )
    .load(
        f"{bronze_base}/latest_trilateration"
    )
)

In [0]:
trip_df = trip_df.dropDuplicates()
trip_schedule_df = trip_schedule_df.dropDuplicates()
tourist_df = tourist_df.dropDuplicates()
agency_df = agency_df.dropDuplicates()
trip_request_places_df = trip_request_places_df.dropDuplicates()
tourist_place_df = tourist_place_df.dropDuplicates()

In [0]:
invalid_location_df = location_stream_df.filter(
    col("TouristId").isNull() |
    (~col("EstimatedLat").between(-90, 90)) |
    (~col("EstimatedLng").between(-180, 180)) |
    (col("Accuracy") <= 0) |
    col("UpdatedAt").isNull()
)

In [0]:
invalid_query = (
    invalid_location_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        f"{quarantine_base}/chk_invalid_location"
    )
    .trigger(availableNow=True)
    .start(
        f"{quarantine_base}/invalid_location"
    )
)

In [0]:
clean_df = location_stream_df.filter(
    col("TouristId").isNotNull() &
    col("EstimatedLat").between(-90, 90) &
    col("EstimatedLng").between(-180, 180) &
    (col("Accuracy") > 0) &
    col("UpdatedAt").isNotNull()
)

In [0]:
clean_df = clean_df.withColumn(
    "UpdatedAt",
    to_timestamp(col("UpdatedAt"))
)

In [0]:
clean_df = clean_df.withWatermark(
    "UpdatedAt",
    "10 minutes"
).dropDuplicates(
    ["TouristId", "UpdatedAt"]
)

In [0]:
trip_df = trip_df.dropna(
    subset=["TripId", "TouristId", "AgencyId"]
)
trip_df = trip_df.fillna({
    "AssignedEmployeeId": 0,
    "Status": "UNKNOWN",
    "RequestId": 0
})

In [0]:
trip_schedule_df = trip_schedule_df.dropna(
    subset=["ScheduleId", "TripRequestId", "PlaceId"]
)

tourist_df = tourist_df.dropna(
    subset=["TouristId", "Name"]
)

agency_df = agency_df.dropna(
    subset=["AgencyId", "AgencyName"]
)

tourist_place_df = tourist_place_df.dropna(
    subset=["PlaceId", "Latitude", "Longitude"]
)

In [0]:
trip_df = trip_df.filter(
    col("Status").isin("APPROVED", "ONGOING")
)

In [0]:
trip_df = trip_df.dropDuplicates(
    ["TripId"]
)
trip_schedule_df = trip_schedule_df.dropDuplicates(
    ["ScheduleId"]
)
agency_df = agency_df.dropDuplicates(
    ["AgencyId"]
)
tourist_df = tourist_df.dropDuplicates(
    ["TouristId"]
)
trip_request_places_df = trip_request_places_df.dropDuplicates(
    ["RequestId", "PlaceId"]
)
tourist_place_df = tourist_place_df.dropDuplicates(
    ["PlaceId"]
)


In [0]:
tourist_place_df = tourist_place_df.withColumn(
    "place_name",
    initcap(col("PlaceName"))
)

In [0]:
trip_master_df = (
    trip_df.select(
        "TripId",
        "TouristId",
        "AgencyId",
        "RequestId",
        col("Status").alias("trip_status"),
        "StartDate",
        "EndDate"
    )
    .join(
        tourist_df.select(
            "TouristId",
            "Name",
            "Nationality"
        ),
        "TouristId",
        "inner"
    )
    .join(
        agency_df.select(
            "AgencyId",
            "AgencyName"
        ),
        "AgencyId",
        "inner"
    )
)
trip_s = trip_schedule_df.select(
    "ScheduleId",
    "TripRequestId",
    "PlaceId",
    col("Status").alias("schedule_status"),
    "ScheduledDate"
).alias("trip_s")

req_p = trip_request_places_df.select(
    "RequestId",
    col("PlaceId").alias("request_place_id")
).alias("req_p")


place_p = tourist_place_df.select(
    "PlaceId",
    "place_name",
    "Latitude",
    "Longitude"
).alias("place_p")

schedule_master_df = (
    trip_s
    .join(
        req_p,
        (
            col("trip_s.TripRequestId") == col("req_p.RequestId")
        ) &
        (
            col("trip_s.PlaceId") == col("req_p.request_place_id")
        ),
        "inner"
    )
    .drop(
        "RequestId",
        "request_place_id"
    )
    .join(
        place_p,
        "PlaceId",
        "inner"
    )
)
master_df = (
    trip_master_df.alias("trip_m")
    .join(
        schedule_master_df.alias("sched_m"),
        col("trip_m.RequestId") == col("sched_m.TripRequestId"),
        "inner"
    )
    .drop("TripRequestId")
)

In [0]:
silver_df = clean_df.join(
    master_df,
    "TouristId",
    "inner"
)

In [0]:
silver_df = silver_df.withColumn(
    "distance_km",
    sqrt(
        pow(col("EstimatedLat") - col("Latitude"), 2) +
        pow(col("EstimatedLng") - col("Longitude"), 2)
    ) * 111
)

In [0]:
silver_df = silver_df.withColumn(
    "movement_status",
    when(
        col("distance_km") <= 0.5,
        "ARRIVED"
    ).otherwise("NOT_ARRIVED")
)

In [0]:
silver_df = silver_df.withColumn(
    "ScheduledDate",
    to_timestamp(col("ScheduledDate"))
)

In [0]:
silver_df = silver_df.withColumn(
    "delay_minutes",
    (
        unix_timestamp(col("UpdatedAt")) -
        unix_timestamp(col("ScheduledDate"))
    ) / 60
)

In [0]:
silver_df = silver_df.withColumn(
    "delay_category",
    when(col("delay_minutes") <= 0, "ON_TIME")
    .when(col("delay_minutes") <= 30, "MINOR_DELAY")
    .otherwise("MAJOR_DELAY")
)

In [0]:
silver_df = silver_df.withColumn(
    "trip_duration_days",
    datediff(
        col("EndDate"),
        col("StartDate")
    )
)

In [0]:
tourist_trip_df = trip_df.groupBy(
    "TouristId"
).agg(
    countDistinct("TripId").alias("tourist_trip_count")
)

In [0]:
agency_tourist_df = trip_df.groupBy(
    "AgencyId"
).agg(
    countDistinct("TouristId").alias("agency_tourist_count")
)

In [0]:
agency_trip_df = trip_df.groupBy(
    "AgencyId"
).agg(
    countDistinct("TripId").alias("agency_trip_count")
)

In [0]:
place_request_df = trip_request_places_df.groupBy(
    "PlaceId"
).agg(
    count("*").alias("place_request_count")
)

In [0]:
silver_df = (
    silver_df
    .join(tourist_trip_df, "TouristId", "left")
    .join(agency_tourist_df, "AgencyId", "left")
    .join(agency_trip_df, "AgencyId", "left")
    .join(place_request_df, "PlaceId", "left")
)

In [0]:
silver_query = (
    silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://checkpoints@travelappprojectstorage.dfs.core.windows.net/trip_adherence_silver_temp"
    )
    .trigger(availableNow=True)
    .start(
        f"{silver_base}/trip_adherence_temp"
    )
)

silver_query.awaitTermination()

In [0]:
from delta.tables import DeltaTable
updates_df = spark.read.format("delta").load(
    f"{silver_base}/trip_adherence_temp"
)

try:
    silver_table = DeltaTable.forPath(
        spark,
        f"{silver_base}/trip_adherence_clean"
    )

    silver_table.alias("target").merge(
        updates_df.alias("source"),
        """
        target.TouristId = source.TouristId
        AND target.TripId = source.TripId
        AND target.ScheduleId = source.ScheduleId
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

except:
    updates_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(
            f"{silver_base}/trip_adherence_clean"
        )

In [0]:
silver_check_df = spark.read.format("delta").load(
    f"{silver_base}/trip_adherence_clean"
)

display(silver_check_df)